**Cell 1**

# NB11 — Train TinyLlama HelpSteer2 DPO Experts

Trains five independent LoRA experts with Direct Preference Optimization (DPO), one per HelpSteer2 attribute. The training pairs are formed **within each prompt**: the response with the higher target-attribute rating is `chosen`, the lower-rated response is `rejected`, and target ties are discarded.

All experts use the same immutable TinyLlama base revision, the same 2,005-pair budget, seed, optimizer, DPO beta, one fixed final epoch, and LoRA architecture. Only the target attribute changes. ArmoRM is never loaded by the training script and cannot affect pair construction, optimization, resumption, or checkpoint selection. This makes the adapters suitable for a non-circular RQ2 regime.

The notebook uses the two supplied programs in repository-local form:

- `scripts/0_dpo_expert.py` trains one DPO expert.
- `scripts/diff_experts.py` performs a strictly post-hoc ArmoRM differentiation check after all five fixed-epoch adapters exist. Its output must not be used to tune training.

`Run all` works sequentially. Every completed axis is copied to Google Drive and exact reruns are skipped; incompatible cached runs fail closed.

**Cell 2**

## 1. Clone or update the repository

In [ ]:
# Cell 3
%cd /content
import os, shutil
repo_path = "/content/master-thesis"
repo_url = "https://github.com/NZhang137/master-thesis.git"
if os.path.isdir(os.path.join(repo_path, ".git")):
    %cd /content/master-thesis
    !git pull
else:
    if os.path.exists(repo_path):
        shutil.rmtree(repo_path)
    !git clone {repo_url} {repo_path}
    %cd /content/master-thesis

**Cell 4**

## 2. Check the GPU

A single A100 is ample. DPO holds TinyLlama and its PEFT reference path, but never ArmoRM, during training. The post-hoc script first unloads every policy and only then loads ArmoRM.

In [ ]:
# Cell 5
!nvidia-smi

**Cell 6**

## 3. Install dependencies

This stack has binary wheels for the current Colab Python 3.13 runtime and supports the DPOTrainer API used by NB11. Run this before importing Transformers, PEFT, or TRL.

In [ ]:
# Cell 7
import sys
print(f"Python {sys.version.split()[0]}")
!pip uninstall -y torchao >/dev/null 2>&1 || true
!pip install -q -U --only-binary=:all: "pandas==2.2.3" "numpy==2.1.3" "protobuf==5.29.5" "transformers==4.46.3" "tokenizers==0.20.3" "peft==0.13.2" "accelerate==1.1.1" "trl==0.11.4" "datasets==3.1.0" "huggingface_hub==0.36.0" bitsandbytes pyyaml safetensors
!python -c "import platform,transformers,tokenizers,peft,accelerate,trl; print('Runtime:',platform.python_version(),'transformers',transformers.__version__,'tokenizers',tokenizers.__version__,'peft',peft.__version__,'accelerate',accelerate.__version__,'trl',trl.__version__)"

**Cell 8**

## 4. Settings, immutable revisions, and Drive persistence

The first run resolves the exact Hugging Face commits and stores them in Drive. Every later run reuses those revisions. Completed experts are restored from Drive before training; their manifests and adapter SHA256 hashes are checked by the scripts.

In [ ]:
# Cell 9
from pathlib import Path
import json, os, shutil
from huggingface_hub import HfApi
from google.colab import drive

PROJECT_ROOT = Path("/content/master-thesis")
RUN_TAG = "nb11_run1"
BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
DATASET_NAME = "nvidia/HelpSteer2"
ATTRIBUTES = ["helpfulness", "correctness", "coherence", "complexity", "verbosity"]
OUTPUT_ROOT = PROJECT_ROOT / "results" / "dpo_rq2" / RUN_TAG
DRIVE_ROOT = Path("/content/drive/MyDrive/master-thesis-nb11") / RUN_TAG
REVISION_FILE = DRIVE_ROOT / "source_revisions.json"
MAX_PAIRS = 2005
DPO_BETA = 0.1
EPOCHS = 1.0
LEARNING_RATE = 5e-4
BATCH_SIZE = 2
GRAD_ACCUM = 4
MAX_LENGTH = 512
MAX_PROMPT_LENGTH = 256
SEED = 8888
RUN_SMOKE_TEST = True
RUN_POSTHOC_EVALUATION = True
EVAL_PROMPTS = 64

drive.mount("/content/drive")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

if REVISION_FILE.exists():
    revisions = json.loads(REVISION_FILE.read_text())
else:
    api = HfApi()
    revisions = {
        "base_model": BASE_MODEL,
        "base_revision": api.model_info(BASE_MODEL).sha,
        "dataset": DATASET_NAME,
        "dataset_revision": api.dataset_info(DATASET_NAME).sha,
    }
    assert revisions["base_revision"] and revisions["dataset_revision"]
    REVISION_FILE.write_text(json.dumps(revisions, indent=2, sort_keys=True) + "\n")

assert revisions["base_model"] == BASE_MODEL
assert revisions["dataset"] == DATASET_NAME
BASE_REVISION = revisions["base_revision"]
DATASET_REVISION = revisions["dataset_revision"]

for axis in ATTRIBUTES:
    source = DRIVE_ROOT / f"dpo_{axis}"
    target = OUTPUT_ROOT / f"dpo_{axis}"
    if source.exists() and not target.exists():
        shutil.copytree(source, target)
        print(f"[restore] {axis} from Drive")

print(f"output             = {OUTPUT_ROOT}")
print(f"Drive backup       = {DRIVE_ROOT}")
print(f"base revision      = {BASE_REVISION}")
print(f"dataset revision   = {DATASET_REVISION}")
print(f"budget             = {MAX_PAIRS} pairs per axis, {EPOCHS} fixed epoch")
print(f"effective batch    = {BATCH_SIZE * GRAD_ACCUM}")
print(f"post-hoc ArmoRM    = {RUN_POSTHOC_EVALUATION} (never use it for tuning)")

**Cell 10**

## 5. Validate the central experiment configuration

In [ ]:
# Cell 11
!python scripts/validate_tinyllama_helpsteer2_config.py
from src.experiment_config import get_attribute_order, load_experiment_config
cfg = load_experiment_config(PROJECT_ROOT / "configs/tinyllama_helpsteer2_armorm.yaml")
assert list(get_attribute_order(cfg)) == ATTRIBUTES
assert cfg["base_model_name"] == BASE_MODEL
assert cfg["dataset_name"] == DATASET_NAME
print("[OK] Base model, dataset, and five-axis order match the central config.")

**Cell 12**

## 6. Inspect all five preference-pair pools

This performs no training and does not load ArmoRM. It verifies that every attribute has at least 2,005 non-tied within-prompt pairs before any GPU time is spent.

In [ ]:
# Cell 13
import importlib.util
from datasets import load_dataset

spec = importlib.util.spec_from_file_location("nb11_dpo_script", PROJECT_ROOT / "scripts/0_dpo_expert.py")
dpo_script = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(dpo_script)
train_rows = load_dataset(DATASET_NAME, split="train", revision=DATASET_REVISION)
for axis in ATTRIBUTES:
    candidates = dpo_script.build_pairs_from_rows(train_rows, axis)
    selected = dpo_script.select_pairs(candidates, seed=SEED, max_pairs=MAX_PAIRS)
    print(f"{axis:12} candidates={len(candidates):5d} selected={len(selected):4d} unique_prompts={len({p['raw_prompt'] for p in selected}):4d}")
print("[OK] Equal-N pair budget is feasible for all axes.")

**Cell 14**

## 7. Evaluator-free DPO smoke test

A 16-pair throwaway run catches TRL, tokenizer, gradient-checkpointing, and VRAM problems before the five real jobs. It is deleted immediately and never becomes an expert.

In [ ]:
# Cell 15
import subprocess, shutil, sys, os
from collections import deque

def run_logged_subprocess(command, label):
    """Stream child output and preserve its final lines in the raised error."""
    print(f"[run] {label}: {' '.join(map(str, command))}", flush=True)
    tail = deque(maxlen=80)
    process = subprocess.Popen(
        command,
        cwd=str(PROJECT_ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
        tail.append(line.rstrip())
    return_code = process.wait()
    if return_code:
        final_output = "\n".join(tail)
        raise RuntimeError(
            f"{label} failed with exit code {return_code}.\n"
            f"Last subprocess output:\n{final_output}"
        )

smoke_root = Path("/tmp/nb11_dpo_smoke")
all_axes_complete = all(
    (OUTPUT_ROOT / f"dpo_{axis}/training_manifest.json").is_file()
    and (OUTPUT_ROOT / f"dpo_{axis}/adapter/adapter_model.safetensors").is_file()
    for axis in ATTRIBUTES
)
if RUN_SMOKE_TEST and not all_axes_complete:
    shutil.rmtree(smoke_root, ignore_errors=True)
    command = [
        sys.executable, "-u", "scripts/0_dpo_expert.py",
        "--reward_name", "helpfulness",
        "--base_model_name", BASE_MODEL,
        "--base_revision", BASE_REVISION,
        "--dataset_name", DATASET_NAME,
        "--dataset_revision", DATASET_REVISION,
        "--output_root", str(smoke_root),
        "--max_pairs", "16",
        "--seed", str(SEED),
        "--epochs", "1",
        "--save_steps", "1000",
        "--overwrite",
    ]
    run_logged_subprocess(command, "DPO smoke test")
    shutil.rmtree(smoke_root, ignore_errors=True)
    print("[OK] Smoke test passed and was deleted.")
elif all_axes_complete:
    print("[skip] All five verified adapter files exist; smoke test is no longer needed.")
else:
    print("Smoke test disabled.")

**Cell 16**

## 8. Training helper

Every call has the same arguments except `reward_name`. `--resume` continues the latest fixed-run trainer checkpoint after a Colab interruption. After successful completion, the complete expert directory is atomically replaced in Drive.

In [ ]:
# Cell 17
import subprocess, shutil, gc, torch

def train_axis(axis):
    assert axis in ATTRIBUTES
    command = [
        sys.executable, "-u", "scripts/0_dpo_expert.py",
        "--reward_name", axis,
        "--base_model_name", BASE_MODEL,
        "--base_revision", BASE_REVISION,
        "--dataset_name", DATASET_NAME,
        "--dataset_revision", DATASET_REVISION,
        "--split", "train",
        "--output_root", str(OUTPUT_ROOT),
        "--beta", str(DPO_BETA),
        "--epochs", str(EPOCHS),
        "--lr", str(LEARNING_RATE),
        "--batch_size", str(BATCH_SIZE),
        "--grad_accum", str(GRAD_ACCUM),
        "--max_length", str(MAX_LENGTH),
        "--max_prompt_length", str(MAX_PROMPT_LENGTH),
        "--max_pairs", str(MAX_PAIRS),
        "--seed", str(SEED),
        "--resume",
    ]
    print(f"\n=== DPO {axis} ===")
    run_logged_subprocess(command, f"DPO training ({axis})")
    source = OUTPUT_ROOT / f"dpo_{axis}"
    destination = DRIVE_ROOT / f"dpo_{axis}"
    temporary = DRIVE_ROOT / f".dpo_{axis}.copying"
    shutil.rmtree(temporary, ignore_errors=True)
    shutil.copytree(source, temporary)
    shutil.rmtree(destination, ignore_errors=True)
    temporary.rename(destination)
    print(f"[backup] {axis} -> {destination}")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

**Cell 18**

### Helpfulness expert

In [ ]:
# Cell 19
train_axis("helpfulness")

**Cell 20**

### Correctness expert

In [ ]:
# Cell 21
train_axis("correctness")

**Cell 22**

### Coherence expert

In [ ]:
# Cell 23
train_axis("coherence")

**Cell 24**

### Complexity expert

In [ ]:
# Cell 25
train_axis("complexity")

**Cell 26**

### Verbosity expert

In [ ]:
# Cell 27
train_axis("verbosity")

**Cell 28**

## 9. Verify and bundle the five adapters

The compact bundle contains adapter files and immutable training manifests, but omits resumable trainer checkpoints. The bundle SHA256 is saved beside it in Drive.

In [ ]:
# Cell 29
import hashlib, json, shutil
bundle_stage = Path("/tmp/nb11_dpo_bundle")
shutil.rmtree(bundle_stage, ignore_errors=True)
bundle_stage.mkdir(parents=True)
shared = {}
for axis in ATTRIBUTES:
    expert = OUTPUT_ROOT / f"dpo_{axis}"
    manifest_path = expert / "training_manifest.json"
    weights_path = expert / "adapter/adapter_model.safetensors"
    assert manifest_path.is_file() and weights_path.is_file(), f"Incomplete expert: {axis}"
    manifest = json.loads(manifest_path.read_text())
    assert manifest["completed"] and manifest["armorm_used_during_training"] is False
    digest = hashlib.sha256(weights_path.read_bytes()).hexdigest()
    assert digest == manifest["adapter_model_sha256"]
    for key in ["trainer_script_sha256", "runtime_versions", "base_revision", "dataset_revision", "selected_pair_count", "beta", "dpo_disable_dropout", "epochs", "learning_rate", "effective_batch_size", "lora"]:
        value = json.dumps(manifest[key], sort_keys=True)
        if key in shared:
            assert shared[key] == value, f"Experts differ on {key}"
        shared[key] = value
    target = bundle_stage / f"dpo_{axis}"
    shutil.copytree(expert / "adapter", target / "adapter")
    shutil.copy2(manifest_path, target / "training_manifest.json")
    print(f"[OK] {axis:12} {digest}")

bundle_base = Path("/content/nb11_tinyllama_helpsteer2_dpo_adapters")
bundle_path = Path(shutil.make_archive(str(bundle_base), "zip", root_dir=bundle_stage))
bundle_sha = hashlib.sha256(bundle_path.read_bytes()).hexdigest()
sha_path = Path(str(bundle_path) + ".sha256")
sha_path.write_text(f"{bundle_sha}  {bundle_path.name}\n")
shutil.copy2(bundle_path, DRIVE_ROOT / bundle_path.name)
shutil.copy2(sha_path, DRIVE_ROOT / sha_path.name)
print(f"bundle = {bundle_path} ({bundle_path.stat().st_size/1e6:.1f} MB)")
print(f"sha256 = {bundle_sha}")

**Cell 30**

## 10. Post-hoc expert differentiation

This executes the second supplied file only after all five fixed adapters have passed verification. It compares the untouched base and all DPO experts on the same 64 held-out validation prompts. ArmoRM is an evaluator here, never a training signal.

The 64 prompt hashes and all generations are saved. Treat this as a consumed diagnostic set: do not tune anything from the output, and exclude these prompts from the later confirmatory RQ2 evaluation.

In [ ]:
# Cell 31
import subprocess, shutil
EVAL_DIR = OUTPUT_ROOT / "posthoc_differentiation"
destination = DRIVE_ROOT / "posthoc_differentiation"
if RUN_POSTHOC_EVALUATION:
    if destination.exists() and not EVAL_DIR.exists():
        shutil.copytree(destination, EVAL_DIR)
        print(f"[restore] evaluation from {destination}")
    command = [
        sys.executable, "-u", "scripts/diff_experts.py",
        "--base_model_name", BASE_MODEL,
        "--base_revision", BASE_REVISION,
        "--expert_root", str(OUTPUT_ROOT),
        "--adapter_subdir", "adapter",
        "--dataset_name", DATASET_NAME,
        "--dataset_revision", DATASET_REVISION,
        "--split", "validation",
        "--n_prompts", str(EVAL_PROMPTS),
        "--prompt_seed", "991",
        "--max_new_tokens", "128",
        "--armorm_precision", "8bit",
        "--output_dir", str(EVAL_DIR),
    ]
    run_logged_subprocess(command, "post-hoc expert differentiation")
    shutil.rmtree(destination, ignore_errors=True)
    shutil.copytree(EVAL_DIR, destination)
    print(f"[backup] evaluation -> {destination}")
else:
    print("Post-hoc evaluation disabled. The five DPO adapters are still complete.")

**Cell 32**

## 11. Read the differentiation result

A useful expert set should show target-specific movement, but this cell is descriptive. A weak result must be reported rather than used to choose another epoch.

In [ ]:
# Cell 33
import json, pandas as pd
report_path = EVAL_DIR / "dpo_expert_report.json"
if report_path.exists():
    report = json.loads(report_path.read_text())
    matrix = pd.read_csv(EVAL_DIR / "dpo_expert_payoff_matrix.csv", index_col=0)
    display(matrix.round(4))
    print(f"Own expert leads its ArmoRM column: {report['n_own_column_leaders']}/{len(ATTRIBUTES)}")
    print("Own-axis change relative to untouched base:")
    for axis, value in report["own_axis_minus_base"].items():
        print(f"  {axis:12} {value:+.4f}")
    print(f"Diagnostic prompt SHA256: {report['prompt_sha256']}")
else:
    print("No post-hoc report because Cell 31 was disabled or has not run.")

**Cell 34**

## 12. Download the adapter bundle

In [ ]:
# Cell 35
from google.colab import files
assert bundle_path.exists() and sha_path.exists()
files.download(str(bundle_path))
files.download(str(sha_path))